In [1]:
!pip install -q transformers peft accelerate tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 84.5 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which 

# End-to-End Paragraph Reconstruction (M-Parallel MLPs + Transformer Decoder)
**Project:** WSAI/032 - Semantic Text Reconstruction from Embeddings

**Architecture Update:**
Instead of a single large MLP projector, this uses **M completely independent MLPs** (where M is `PREFIX_LEN`). 
- Each MLP receives the exact same 1024-dim paragraph embedding.
- Each MLP outputs a single "virtual token".
- These virtual tokens are stacked together and fed into the Qwen Decoder as the prompt.
- Trained end-to-end with Cross-Entropy Loss and LoRA.

---
## Section 1: Setup & Imports

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import h5py
import matplotlib.pyplot as plt
import os
import time
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import get_peft_model, LoraConfig, TaskType

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU: Tesla T4
Memory: 15.6 GB


---
## Section 2: Hyperparameters

In [3]:
H5_PATH = '/kaggle/input/datasets/subhavkumar/new-msmarco-embeddings-pair-dataset/corrected_msmarco_token_sentence_embeddings.h5'

# Fallback path finder
if not os.path.exists(H5_PATH):
    for root, dirs, files in os.walk('/kaggle/input'):
        for fname in files:
            if fname.endswith('.h5'):
                H5_PATH = os.path.join(root, fname)
                break

# Fixed typo in the model name
MODEL_NAME = 'Qwen/Qwen3-0.6B'  

PREFIX_LEN = 32          # Number of M-Parallel MLPs (Virtual Tokens)
MAX_TEXT_LEN = 128       # Max length of actual text tokens to train on
PARAGRAPH_DIM = 1024     # Size of input paragraph embeddings

BATCH_SIZE = 8           
EPOCHS = 5             # Set back to 10 so it trains properly
LEARNING_RATE = 2e-4
TRAIN_SPLIT = 0.90
PATIENCE = 3

print(f'Using {MODEL_NAME} as the Decoder.')
print(f'Using {PREFIX_LEN} Parallel MLPs to map Paragraph Embedding -> Virtual Tokens.')

Using Qwen/Qwen3-0.6B as the Decoder.
Using 32 Parallel MLPs to map Paragraph Embedding -> Virtual Tokens.


---
## Section 3: Load Tokenizer & Dataset

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

class EndToEndDataset(Dataset):
    def __init__(self, h5_path, max_text_len):
        print('Loading dataset...')
        f = h5py.File(h5_path, 'r')
        num = f['seq_lengths'].shape[0]
        
        self.paragraph_embs = []
        self.input_ids = []
        self.attention_masks = []
        self.texts = []
        
        for i in range(num):
            sent_emb = np.array(f['sentence_embeddings'][i], dtype=np.float32)
            ids = np.array(f['input_ids'][i], dtype=np.int64)
            seq_len = int(f['seq_lengths'][i])
            text = f['texts'][i]
            if isinstance(text, bytes): text = text.decode('utf-8')
            
            actual_len = min(seq_len, max_text_len)
            padded_ids = np.full(max_text_len, tokenizer.pad_token_id, dtype=np.int64)
            padded_ids[:actual_len] = ids[:actual_len]
            
            mask = np.zeros(max_text_len, dtype=np.float32)
            mask[:actual_len] = 1.0
            
            self.paragraph_embs.append(sent_emb)
            self.input_ids.append(padded_ids)
            self.attention_masks.append(mask)
            self.texts.append(text)
            
        f.close()
        self.paragraph_embs = torch.tensor(np.array(self.paragraph_embs))
        self.input_ids = torch.tensor(np.array(self.input_ids))
        self.attention_masks = torch.tensor(np.array(self.attention_masks))
        print(f'Done! Embs={self.paragraph_embs.shape}, IDs={self.input_ids.shape}')
    
    def __len__(self): return len(self.paragraph_embs)
    def __getitem__(self, idx):
        return self.paragraph_embs[idx], self.input_ids[idx], self.attention_masks[idx]

full_dataset = EndToEndDataset(H5_PATH, MAX_TEXT_LEN)
train_size = int(TRAIN_SPLIT * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Loading dataset...
Done! Embs=torch.Size([10000, 1024]), IDs=torch.Size([10000, 128])


---
## Section 4: Architecture (M-Parallel MLPs)

In [5]:
class SingleTokenMLP(nn.Module):
    def __init__(self, paragraph_dim, decoder_hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(paragraph_dim, 2048),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(2048, decoder_hidden_dim)
        )
    def forward(self, x):
        return self.net(x)


class EndToEndInverter(nn.Module):
    def __init__(self, paragraph_dim, decoder_hidden_dim, prefix_len, decoder_model):
        super().__init__()
        self.prefix_len = prefix_len
        self.decoder_hidden_dim = decoder_hidden_dim
        
        # M completely independent MLPs (no shared weights)
        self.m_parallel_mlps = nn.ModuleList([
            SingleTokenMLP(paragraph_dim, decoder_hidden_dim)
            for _ in range(prefix_len)
        ])
        
        self.decoder = decoder_model
        
    def forward(self, paragraph_embs, text_input_ids, attention_mask):
        batch_size = paragraph_embs.shape[0]
        
        # 1. Feed paragraph embedding into all M MLPs in parallel
        outputs = [mlp(paragraph_embs) for mlp in self.m_parallel_mlps]
        
        # Stack them together -> [batch, prefix_len, hidden_dim]
        prefix_embeds = torch.stack(outputs, dim=1) 
        
        # 2. Get embeddings for actual ground-truth text
        text_embeds = self.decoder.get_input_embeddings()(text_input_ids)  
        
        # 3. Cast to bfloat16 to match the decoder, then concatenate
        prefix_embeds = prefix_embeds.to(text_embeds.dtype)
        inputs_embeds = torch.cat([prefix_embeds, text_embeds], dim=1)  
        
        # 4. Create labels (-100 for prefix tokens so loss ignores them)
        prefix_labels = torch.full((batch_size, self.prefix_len), -100, dtype=torch.long).to(text_input_ids.device)
        labels = torch.cat([prefix_labels, text_input_ids], dim=1) 
        
        # 5. Create Attention Mask 
        prefix_mask = torch.ones((batch_size, self.prefix_len), dtype=attention_mask.dtype).to(attention_mask.device)
        concat_mask = torch.cat([prefix_mask, attention_mask], dim=1) 
        
        # 6. Forward pass through decoder
        outputs = self.decoder(inputs_embeds=inputs_embeds, attention_mask=concat_mask, labels=labels)
        return outputs.loss
        
    def generate(self, paragraph_embs, tokenizer, max_new_tokens=128):
        batch_size = paragraph_embs.shape[0]
        
        # Generate virtual tokens through parallel MLPs
        outputs = [mlp(paragraph_embs) for mlp in self.m_parallel_mlps]
        prefix_embeds = torch.stack(outputs, dim=1)
        prefix_embeds = prefix_embeds.to(self.decoder.dtype)
        
        # Auto-regressively generate text
        outputs = self.decoder.generate(
            inputs_embeds=prefix_embeds,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=False,  
        )
        return tokenizer.batch_decode(outputs, skip_special_tokens=True)

In [6]:
# Load Decoder (device_map={'': device} forces everything onto GPU 0 to prevent splitting errors)
print(f'Loading base decoder {MODEL_NAME}...')
base_decoder = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map={'': device}
)

# Apply LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"]
)
decoder_with_lora = get_peft_model(base_decoder, lora_config)
decoder_hidden_size = base_decoder.config.hidden_size

# Initialize model (no .to(device) here, we move the MLPs manually)
model = EndToEndInverter(
    paragraph_dim=PARAGRAPH_DIM,
    decoder_hidden_dim=decoder_hidden_size,
    prefix_len=PREFIX_LEN,
    decoder_model=decoder_with_lora
)
model.m_parallel_mlps.to(device)

model.train()
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nModel initialized!')
print(f'Decoder hidden size: {decoder_hidden_size}')
print(f'Trainable params: {trainable_params:,} ({PREFIX_LEN} MLPs + LoRA Adapters)')

Loading base decoder Qwen/Qwen3-0.6B...


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]


Model initialized!
Decoder hidden size: 1024
Trainable params: 136,609,792 (32 MLPs + LoRA Adapters)


---
## Section 5: Training Loop (with TQDM)

In [7]:
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {'train_loss': [], 'test_loss': []}
best_loss = float('inf')
patience_counter = 0
SAVE_PATH = '/kaggle/working/best_end_to_end_model.pt'

print(f'Training for {EPOCHS} epochs...')
print('='*60)

for epoch in range(EPOCHS):
    t0 = time.time()
    
    # --- TRAIN ---
    model.train()
    train_loss, train_steps = 0, 0
    
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for emb, ids, mask in train_pbar:
        emb, ids, mask = emb.to(device), ids.to(device), mask.to(device)
        
        loss = model(emb, ids, mask)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        train_loss += loss.item()
        train_steps += 1
        train_pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    # --- TEST ---
    model.eval()
    test_loss, test_steps = 0, 0
    
    test_pbar = tqdm(test_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Test]", leave=False)
    with torch.no_grad():
        for emb, ids, mask in test_pbar:
            emb, ids, mask = emb.to(device), ids.to(device), mask.to(device)
            loss = model(emb, ids, mask)
            
            test_loss += loss.item()
            test_steps += 1
            test_pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
    t_loss_avg = train_loss / train_steps
    v_loss_avg = test_loss / test_steps
    history['train_loss'].append(t_loss_avg)
    history['test_loss'].append(v_loss_avg)
    
    if v_loss_avg < best_loss:
        best_loss = v_loss_avg
        patience_counter = 0
        # Save the MLPs and the LoRA adapters
        torch.save(model.state_dict(), SAVE_PATH)
        flag = ' << BEST'
    else:
        patience_counter += 1
        flag = ''
        
    scheduler.step()
    
    print(f'Epoch {epoch+1:2d}/{EPOCHS} | Train Loss: {t_loss_avg:.4f} | Test Loss: {v_loss_avg:.4f} | {time.time()-t0:.1f}s {flag}')
    
    if patience_counter >= PATIENCE:
        print("\nEarly stopping triggered!")
        break

Training for 5 epochs...


Epoch  1/5 | Train Loss: 1.6240 | Test Loss: 1.5357 | 2072.2s  << BEST


Epoch  2/5 | Train Loss: 1.4359 | Test Loss: 1.4833 | 2079.5s  << BEST


Epoch  3/5 | Train Loss: 1.3149 | Test Loss: 1.4616 | 2081.8s  << BEST


Epoch  4/5 | Train Loss: 1.1992 | Test Loss: 1.4593 | 2080.8s  << BEST


Epoch  5/5 | Train Loss: 1.1062 | Test Loss: 1.4747 | 2072.5s 


---
## Section 6: Inference & Evaluation

In [8]:
# Load best model
model.load_state_dict(torch.load(SAVE_PATH))
model.eval()

print('='*80)
print('END-TO-END GENERATION RESULTS')
print('='*80)

np.random.seed(42)
sample_indices = np.random.choice(len(test_dataset), size=5, replace=False)

with torch.no_grad():
    for idx in sample_indices:
        emb, ids, mask = test_dataset[idx]
        original_text = full_dataset.texts[test_dataset.indices[idx]]
        
        # Generate text directly from the embedding
        emb_batch = emb.unsqueeze(0).to(device)
        generated_text = model.generate(emb_batch, tokenizer, max_new_tokens=64)[0]
        
        print(f'\nSample {idx}:')
        print(f'ORIGINAL : {original_text[:200]}')
        print(f'GENERATED: {generated_text}')
        print('-'*80)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


END-TO-END GENERATION RESULTS

Sample 521:
ORIGINAL : Many Sunrise Senior Living communities have also received the Senior Advisor Excellence Award based on ratings and reviews from consumers. To qualify for an Excellence Award, communities must have rec
GENERATED: The Senior Assessment Rating System (SARS) is a nationally recognized rating system for senior housing. The SARS rating is based on the quality of the community, the amenities, the staff, the services, and the overall experience of residents. The SARS rating is awarded to senior housing.
--------------------------------------------------------------------------------

Sample 737:
ORIGINAL : You may order copies of Colorado vital records through VitalChek on an expedited basis. Colorado Vital Records requires all applicants to submit a copy of proof of relationship (e.g. marriage certific
GENERATED: To obtain a Colorado copy of your marriage certificate, you must provide the following documents: 1  A certified copy of your ma

In [9]:
# Extract only the trained weights from the model currently in memory
trainable_weights = {k: v for k, v in model.state_dict().items() if v.requires_grad}

# Save it as a new, tiny file
torch.save(trainable_weights, '/kaggle/working/tiny_best_model.pt')

print("Tiny model saved successfully!")

Tiny model saved successfully!


In [10]:
# Fixed: Use named_parameters() to keep the requires_grad information!
trainable_weights = {k: v for k, v in model.named_parameters() if v.requires_grad}

# Save it as a new file
torch.save(trainable_weights, '/kaggle/working/tiny_best_model.pt')

print("Model saved successfully! (Should be around 500 MB)")

Model saved successfully! (Should be around 500 MB)


In [11]:
!pip install -q wandb
import wandb

# 1. Put your actual WandB API key inside the quotes below
WANDB_API_KEY = "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" 

# 2. Log in directly using the key
wandb.login(key=WANDB_API_KEY)

# 3. Initialize a new project and run
wandb.init(
    project="Embedding-Inversion",   # The main folder for your research
    name="Run-1-5-Epochs"            # The specific name for this test
)

# 4. Loop through your already-saved history and upload it!
for epoch in range(len(history['train_loss'])):
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": history['train_loss'][epoch],
        "test_loss": history['test_loss'][epoch]
    })

# 5. Close the run and generate the dashboard link
wandb.finish()

print("Historical data successfully uploaded to WandB!")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: subhavpathak18 (subhavpathak18-indian-institute-of-information-technolog) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


epoch,▁▃▅▆█
test_loss,█▃▁▁▂
train_loss,█▅▄▂▁
epoch,5
test_loss,1.47466
train_loss,1.10621


Historical data successfully uploaded to WandB!
